### Integration vectordb context pipeline with LLM output

In [1]:
# Shared RAG imports and LLM setup
import os
import sys
from pathlib import Path

from langchain_groq import ChatGroq

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import GROQ_MAX_TOKENS, GROQ_MODEL, GROQ_TEMPERATURE
from src.embeddings import EmbeddingManager
from src.pipeline import AdvancedRAGPipeline, rag_advanced, rag_simple
from src.retriever import RAGRetriever
from src.vector_store import VectorStoreLoader

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is not set. Add it to your .env file before running this cell.")

llm = ChatGroq(
    api_key=groq_api_key,
    model_name=GROQ_MODEL,
    temperature=GROQ_TEMPERATURE,
    max_tokens=GROQ_MAX_TOKENS,
)


d:\Data Sciece Mastery\RAG Learnings\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
active_retriever = globals().get("rag_retriever", globals().get("retriever"))
if active_retriever is None:
    embedding_manager = EmbeddingManager()
    vector_store = VectorStoreLoader()
    active_retriever = RAGRetriever(vector_store, embedding_manager)

answer = rag_simple("What is the attention mechanism?", active_retriever, llm)
print("Answer:", answer)


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4706.61it/s]


Answer: The attention mechanism is a technique used in neural networks that allows models to focus on important parts of input data. It is widely used in Natural Language Processing and computer vision tasks. Instead of processing all information equally, attention assigns weights to different parts of the input, helping models capture relationships between words or features more effectively. This is particularly useful in tasks such as machine translation, text summarization, sentiment analysis, and question answering.

At its core, attention works by computing relationships between elements of a sequence, involving three main components: queries, keys, and values. A query is compared with a set of keys to produce similarity scores, which are then normalized using a softmax function. These scores are used to compute a weighted sum of the values, resulting in a context-aware representation that emphasizes the most relevant parts of the input.

There are different types of attention mec

### Enhanced RAG Pipeline Features ...

In [3]:
# Enhanced RAG pipeline demo
active_retriever = globals().get("rag_retriever", globals().get("active_retriever", globals().get("retriever")))
if active_retriever is None:
    embedding_manager = EmbeddingManager()
    vector_store = VectorStoreLoader()
    active_retriever = RAGRetriever(vector_store, embedding_manager)

result = rag_advanced(
    "What is the advantage of using attention mechanisms?",
    active_retriever,
    llm,
    top_k=6,
    return_context=True,
)
print("Answer:", result["answer"].strip())
print("Sources:", result["sources"])
print("Confidence Score:", result["confidence_score"])
print("context preview:", result["context"][:300])


Answer: The advantage of using attention mechanisms is that they enable models to process information more efficiently and effectively. Attention provides a solution by allowing the model to dynamically focus on different parts of the input sequence during processing. This is achieved by computing relationships between elements of a sequence, which involves three main components: queries, keys, and values.

The attention mechanism has several benefits, including:

1. **Improved processing efficiency**: By focusing on important parts of the input, attention mechanisms reduce the computational cost of processing large amounts of data.
2. **Enhanced contextual understanding**: Attention allows models to capture relationships between words or features more effectively, leading to a deeper understanding of the input data.
3. **Increased interpretability**: By examining attention weights, researchers can understand which parts of the input the model considers important, providing insights in

In [4]:
# Advanced RAG pipeline demo: streaming, citations, history, and summarization.
adv_rag = AdvancedRAGPipeline(active_retriever, llm)
result = adv_rag.query(
    "What are the benefits of attention mechanisms in deep learning?",
    top_k=6,
    stream=True,
    summarize=True,
)
print("Final Answer:", result["answer"].strip())
print("summary:", result["summary"])
print("history:", result["history"][-1])


Generating answer...
Final Answer: The benefits of attention mechanisms in deep learning include:

1. **Efficient processing of long sequences**: Attention mechanisms allow models to dynamically focus on different parts of the input sequence during processing, solving the limitation of earlier models such as recurrent neural networks (RNNs) and long short-term memory networks (LSTMs) that struggled to handle long sequences effectively.

2. **Improved capture of relationships**: Attention provides a solution by allowing the model to compute relationships between elements of a sequence, enabling it to capture relationships between words or features more effectively.

3. **Multi-tasking capabilities**: Transformers, which power modern Large Language Models (LLMs), rely heavily on attention mechanisms, allowing them to capture different types of relationships simultaneously, such as syntactic and semantic connections.

4. **Improved interpretability**: By examining attention weights, resea